# File Process

1. Check all files for headers and print them
2. Union all files, filtered to Jan 2012 - Dec 2016
3. Keep all columns; missing ones left blank
4. Output to the same data folder

In [1]:
import csv
import glob
import os
import duckdb

# Use relative parameters
DATA_DIR = os.path.join("data", "Raw")
COMBINED_DIR = os.path.join("data", "Combined")

# Ensure the Combined folder exists
os.makedirs(COMBINED_DIR, exist_ok=True)

START_MONTH = "2012-01"
END_MONTH = "2016-12"
OUTPUT_FILE = os.path.join(COMBINED_DIR, "resale_flat_prices_2012_01_to_2016_12.parquet")

In [2]:
# 1. Get all the files first
all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

# 2. Filter out the output file using a simple, standard loop
files_to_process = []
for file in all_files:
    if file != OUTPUT_FILE:
        files_to_process.append(file)
        
# Sort the list alphabetically
files_to_process.sort()

print("Headers per file:")
all_headers = []

# 3. Read the headers in a straightforward way without extra modules
for file_path in files_to_process:
    
    # Open the file in basic read mode
    with open(file_path, 'r') as f:
        # Read only the top line of the text file
        first_line = f.readline()
        
        # Clean up the text and split it by commas to make a list
        header = first_line.strip().split(',')
        
        # Get just the file name for a clean print statement
        file_name = os.path.basename(file_path)
        print(f"  {file_name}: {header}")
        
        all_headers.append(header)

Headers per file:
  ResaleFlatPricesBasedonApprovalDate19901999.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']
  ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
  ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'f

In [3]:
union_columns = []
for header in all_headers:
    for col in header:
        if col not in union_columns:
            union_columns.append(col)

#print("Union of columns:", union_columns)

In [4]:
# Format file paths with forward slashes for DuckDB SQL compatibility
files_input = [f.replace('\\', '/') for f in files_to_process]
print(f" Input files are: {files_input}")
output_parquet = OUTPUT_FILE.replace('\\', '/')
cols_str = ', '.join(union_columns)

# Ensure destination file is clean on Windows before writing
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

# Stream filtered results directly to Parquet, sorted chronologically
duckdb.sql(f"""
    COPY (
        SELECT {cols_str}
        FROM read_csv_auto({files_input}, union_by_name=True)
        WHERE month >= '{START_MONTH}' AND month <= '{END_MONTH}'
        ORDER BY month
    ) TO '{output_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# 1. Verify total row count and file size
file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"Success! Wrote to {OUTPUT_FILE} ({file_size_mb:.2f} MB)\n")


 Input files are: ['data/Raw/ResaleFlatPricesBasedonApprovalDate19901999.csv', 'data/Raw/ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv', 'data/Raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv', 'data/Raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv', 'data/Raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv']


Success! Wrote to data\Combined\resale_flat_prices_2012_01_to_2016_12.parquet (0.59 MB)

